In [ ]:
import sys
sys.path.append("../src")

from energy_load_forecast.ingestion import load_and_resample
from pathlib import Path

import pandas as pd

In [ ]:
series = load_and_resample(Path("../data/raw/entsoe/load_RO_2026-09-13_2026-09-20.csv"))
series_copy = series.copy() 
print(series_copy.head())

timestamp_utc
2026-09-13 00:00:00+00:00    4417.25
2026-09-13 01:00:00+00:00    4406.50
2026-09-13 02:00:00+00:00    4544.25
2026-09-13 03:00:00+00:00    4611.25
2026-09-13 04:00:00+00:00    4719.75
Freq: h, Name: load_mw, dtype: float64


In [23]:
print(series_copy.shape)
print(series_copy.isna().sum())

(168,)
25


In [24]:
series_copy = series_copy.dropna()

expected_index = pd.date_range(
    series_copy.index.min(),
    series_copy.index.max(),
    freq="h",
)

assert expected_index.equals(series_copy.index), (
    "Missing or duplicate hourly timestamps found"
)

print(series_copy.shape)

(143,)


In [25]:
df = series_copy.to_frame()
df_copy = df.copy()
print(df_copy.head())

                           load_mw
timestamp_utc                     
2026-09-13 00:00:00+00:00  4417.25
2026-09-13 01:00:00+00:00  4406.50
2026-09-13 02:00:00+00:00  4544.25
2026-09-13 03:00:00+00:00  4611.25
2026-09-13 04:00:00+00:00  4719.75


In [26]:
local_index = series_copy.tz_convert("Europe/Bucharest")
df_copy["hour_local"] = local_index.index.hour
df_copy["day_of_week_local"] = local_index.index.day_of_week
df_copy["is_weekend_local"] = local_index.index.day_of_week >= 5
df_copy["month_local"] = local_index.index.month

df_copy.head()

,load_mw,hour_local,day_of_week_local,is_weekend_local,month_local
timestamp_utc,,,,,
2026-09-13 00:00:00+00:00,4417.25,3,6,True,9
2026-09-13 01:00:00+00:00,4406.50,4,6,True,9
2026-09-13 02:00:00+00:00,4544.25,5,6,True,9
2026-09-13 03:00:00+00:00,4611.25,6,6,True,9
2026-09-13 04:00:00+00:00,4719.75,7,6,True,9


In [27]:
def chronological_split(
    df: pd.DataFrame,
    train_end: str,
    val_end: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not df.index.is_monotonic_increasing:
        raise ValueError("DataFrame index must be sorted before splitting")

    train_end_ts = pd.Timestamp(train_end, tz="UTC")
    val_end_ts = pd.Timestamp(val_end, tz="UTC")

    if not (df.index.min() < train_end_ts < val_end_ts < df.index.max()):
        raise ValueError(
            "Split boundaries must satisfy: data_start < train_end < "
            f"val_end < data_end. Got train_end={train_end_ts}, "
            f"val_end={val_end_ts}, data range=[{df.index.min()}, "
            f"{df.index.max()}]"
        )

    train = df.loc[df.index < train_end_ts]
    val = df.loc[(df.index >= train_end_ts) & (df.index < val_end_ts)]
    test = df.loc[df.index >= val_end_ts]

    return train, val, test

In [28]:
train, val, test = chronological_split(df, "2026-09-16", "2026-09-18")
train.shape, val.shape, test.shape

((72, 1), (48, 1), (23, 1))

In [29]:
train.index.max(), val.index.min(), val.index.max(), test.index.min()

(Timestamp('2026-09-15 23:00:00+0000', tz='UTC'),
 Timestamp('2026-09-16 00:00:00+0000', tz='UTC'),
 Timestamp('2026-09-17 23:00:00+0000', tz='UTC'),
 Timestamp('2026-09-18 00:00:00+0000', tz='UTC'))